In [ ]:
!pip install transformers accelerate bitsandbytes faiss-cpu sentence-transformers langdetect

In [ ]:
import torch
import faiss
import numpy as np
import pickle
import os

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from langdetect import detect

In [ ]:
model_name = "Qwen/Qwen2-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True,
    torch_dtype=torch.float16
)

In [ ]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

dim = 384
index = faiss.IndexFlatL2(dim)

memory_texts = []

# load nếu có
if os.path.exists("faiss.index"):
    index = faiss.read_index("faiss.index")

if os.path.exists("memory.pkl"):
    with open("memory.pkl", "rb") as f:
        memory_texts = pickle.load(f)

In [ ]:
def add_memory(text):
    vec = embed_model.encode([text])
    index.add(np.array(vec).astype("float32"))
    memory_texts.append(text)

def retrieve(text, k=3):
    if len(memory_texts) < k:
        return memory_texts

    vec = embed_model.encode([text])
    D, I = index.search(np.array(vec).astype("float32"), k)

    return [memory_texts[i] for i in I[0]]

In [ ]:
LANG_MAP = {
    "zh-cn": "Chinese",
    "zh-tw": "Chinese",
    "ko": "Korean",
    "en": "English",
    "vi": "Vietnamese"
}

def detect_lang(text):
    try:
        return LANG_MAP.get(detect(text), "Unknown")
    except:
        return "Unknown"

In [ ]:
def split_text(text, max_len=400):
    lines = text.split("\n")
    chunks, cur = [], ""

    for line in lines:
        if len(cur) + len(line) < max_len:
            cur += line + "\n"
        else:
            chunks.append(cur.strip())
            cur = line + "\n"

    if cur:
        chunks.append(cur.strip())

    return chunks

In [ ]:
GLOSSARY = {
    "林凡": "Lâm Phàm",
    "斗气": "Đấu Khí"
}

In [ ]:
def build_prompt(text, context, retrieved, src, tgt):
    glossary_text = "\n".join([f"{k} = {v}" for k,v in GLOSSARY.items()])
    ref = "\n".join(retrieved)

    return f"""
You are a professional novel translator.

Translate from {src} to {tgt}.

Requirements:
- Natural, fluent
- Keep tone & personality
- Use novel style

Glossary:
{glossary_text}

Context:
{context}

Reference:
{ref}

Text:
{text}

Translation:
"""

In [ ]:
def translate_chunk(chunk, context, src, tgt):
    retrieved = retrieve(chunk)

    prompt = build_prompt(chunk, context, retrieved, src, tgt)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.7
    )

    out = tokenizer.decode(outputs[0], skip_special_tokens=True)

    add_memory(out)

    return out

In [ ]:
def process_txt(input_path, output_path, target_lang="Vietnamese"):
    with open(input_path, "r", encoding="utf-8") as f:
        text = f.read()

    src = detect_lang(text[:1000])
    print("Detected:", src)

    chunks = split_text(text)
    results = []
    context = ""

    for i, chunk in enumerate(chunks):
        print(f"Chunk {i+1}/{len(chunks)}")

        out = translate_chunk(chunk, context, src, target_lang)

        results.append(out)
        context = out[-500:]

    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n\n".join(results))

In [ ]:
def save_system():
    faiss.write_index(index, "faiss.index")

    with open("memory.pkl", "wb") as f:
        pickle.dump(memory_texts, f)

In [ ]:
INPUT_FILE = "/content/input.txt"
OUTPUT_FILE = "/content/output.txt"

process_txt(INPUT_FILE, OUTPUT_FILE)

save_system()